# Modern OpenTelemetry logging in Elastic: Python, Node.js, Java, Go and .NET

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/opentelemetry-logging-elastic).
Run the cells in order.

Requires Elastic Cloud Hosted 9.2 or later (or Observability Serverless), Docker with Compose, and a
`.env` copied from `.env.example`. To create the API key, in Kibana **Dev Tools**:

```
POST /_security/api_key
{ "name": "opentelemetry-logging-elastic-notebook" }
```

In [ ]:
%pip install -q --disable-pip-version-check -r requirements.txt

## 1. Connect

In [ ]:
import os
import subprocess
import time

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv(override=True)

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"

# Read by docker-compose.yml.
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = os.environ[
    "OTEL_EXPORTER_OTLP_ENDPOINT"
].rstrip("/")
os.environ["OTLP_API_KEY"] = os.getenv("OTLP_API_KEY") or API_KEY

# Everything this notebook creates. Clean up deletes exactly these.
STREAM = "logs-modern_logging.otel-lab"
SERVICES = [
    "checkout-python",
    "checkout-node",
    "checkout-java",
    "checkout-go",
    "checkout-dotnet",
]

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "elastic-api-version": "2023-10-31",
    "Content-Type": "application/json",
}


def kbn(method, path, **kwargs):
    response = requests.request(
        method, f"{KBN_BASE}{path}", headers=KBN_HEADERS, timeout=120, **kwargs
    )
    if not response.ok:
        print(response.status_code, response.text[:1000])
    response.raise_for_status()
    return response.json() if response.text else {}


def esql(query):
    print(es.esql.query(query=query, format="txt").body)


def compose(*args):
    result = subprocess.run(
        ["docker", "compose", *args], capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"docker compose {' '.join(args)} failed:\n{result.stderr[-3000:]}"
        )
    return result


def set_processing(steps):
    ingest = kbn("GET", f"/api/streams/{STREAM}/_ingest")["ingest"]
    ingest["processing"] = {"steps": steps}
    kbn("PUT", f"/api/streams/{STREAM}/_ingest", json={"ingest": ingest})


def delete_stream():
    if es.indices.exists(index=STREAM):
        set_processing([])
        es.indices.delete_data_stream(name=STREAM)


def run_programs():
    for service in SERVICES:
        compose("run", "--rm", service)
        print(f"{service}: sent")


def wait_for_count(expected, timeout=180):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if es.indices.exists(index=STREAM):
            es.indices.refresh(index=STREAM)
            count = es.count(index=STREAM)["count"]
            if count >= expected:
                print(f"{STREAM}: {count} documents")
                return
        time.sleep(5)
    raise TimeoutError(f"{STREAM} did not reach {expected} documents in {timeout} s")


print(f"Elasticsearch {es.info()['version']['number']}")
print(f"Kibana {kbn('GET', '/api/status')['version']['number']}")
print(compose("version").stdout.strip())

## 2. Build the five programs

The first build takes a few minutes.

In [ ]:
compose("build")
print("Images built:", ", ".join(SERVICES))

## 3. Send the same events from five languages

Starts from an empty stream, so the cell can be run again.

In [ ]:
delete_stream()
run_programs()
wait_for_count(15)

## 4. Check what arrived in Elastic

In [ ]:
esql(
    f"""
FROM {STREAM}
| WHERE severity_number >= 17
| KEEP service.name, severity_text, severity_number, body.text, attributes.app.order.id, attributes.app.amount
| SORT service.name
| LIMIT 10
"""
)

esql(
    f"""
FROM {STREAM}
| STATS records = COUNT() BY service.name
| KEEP service.name, records
| SORT service.name
| LIMIT 10
"""
)

caps = es.field_caps(index=STREAM, fields=["attributes.app.*"])["fields"]
for field, types in sorted(caps.items()):
    if "object" not in types:
        print(f"{field}: {', '.join(types)}")

## 5. Add context without changing the application

The same **Set** processor as in the article, through the Streams API. Then a second batch, because
processing only applies to new records.

In [ ]:
set_processing(
    [
        {
            "action": "set",
            "to": "attributes.app.team",
            "value": "checkout",
            "ignore_failure": False,
        }
    ]
)
run_programs()
wait_for_count(30)

esql(
    f"""
FROM {STREAM}
| STATS records = COUNT(), with_team = COUNT(attributes.app.team) BY service.name
| SORT service.name
| LIMIT 10
"""
)

## 6. Query the five languages with ES|QL

In [ ]:
esql(
    f"""
FROM {STREAM}
| WHERE severity_number >= 17 AND attributes.app.team == "checkout"
| EVAL service = service.name, order_id = attributes.app.order.id,
    amount = attributes.app.amount, team = attributes.app.team
| KEEP service, order_id, amount, team
| SORT service
| LIMIT 1000
"""
)

## 7. Clean up

Deletes the stream and its processor, and the five local images.

In [ ]:
delete_stream()
compose("down", "--rmi", "local")
print("Clean up done")